In [ ]:
pip install nltk scikit-learn pandas numpy matplotlib seaborn

In [ ]:

import random,pickle,os
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

lemmatizer=WordNetLemmatizer()
stop_words=set(stopwords.words('english'))

data={
"greeting":["hello","hi","hey","good morning","good evening","greetings","howdy"],
"farewell":["bye","goodbye","see you","take care","quit","exit"],
"admission":["how to apply","admission process","application form","enrollment process","registration process","how can i join","admission date"],
"fees":["fees","fee structure","tuition fee","semester fee","annual fee","college fee","payment details"],
"courses":["courses","degrees","available programs","departments","computer science","mba","mca"],
"hostel":["hostel","accommodation","dormitory","room facility","where can i stay","girls hostel","boys hostel"],
"placements":["placement","salary package","job opportunities","campus recruitment","highest package","average package","internships"],
"thanks":["thanks","thank you","awesome","great","helpful"],
"complaint":["bad service","poor service","not satisfied","unhappy","issue","problem","disappointed"]
}

responses={k:[f"Response for {k} intent."] for k in data}
responses["greeting"]=["Hello! Welcome to CollegeBot."]
responses["farewell"]=["Goodbye!"]
responses["complaint"]=["Sorry to hear that. Please explain your issue."]

def advanced_preprocess(text):
    tokens=word_tokenize(text.lower())
    tokens=[t for t in tokens if t.isalpha()]
    tokens=[t for t in tokens if t not in stop_words]
    tokens=[lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

X_proc=[]; y=[]
for tag,patterns in data.items():
    for p in patterns:
        X_proc.append(advanced_preprocess(p))
        y.append(tag)

tfidf=TfidfVectorizer(ngram_range=(1,2))
X_tfidf=tfidf.fit_transform(X_proc)

le=LabelEncoder()
y_enc=le.fit_transform(y)

X_train,X_test,y_train,y_test=train_test_split(X_tfidf,y_enc,test_size=0.2,random_state=42)

model=LogisticRegression(max_iter=500)
model.fit(X_train,y_train)

pred=model.predict(X_test)
print("Accuracy:",accuracy_score(y_test,pred))
print(classification_report(y_test,pred))

cm=confusion_matrix(y_test,pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt='d')
plt.savefig("chatbot_confusion_matrix.png")

pickle.dump(tfidf,open("chatbot_tfidf.pkl","wb"))
pickle.dump(model,open("chatbot_model.pkl","wb"))

def load_model():
    tfidf=pickle.load(open("chatbot_tfidf.pkl","rb"))
    model=pickle.load(open("chatbot_model.pkl","rb"))
    return tfidf,model

def predict_intent_cosine(user_input,threshold=0.25):
    processed=advanced_preprocess(user_input)
    vec=tfidf.transform([processed])
    sims=cosine_similarity(vec,X_tfidf)[0]
    idx=sims.argmax()
    score=sims[idx]
    tag=y[idx]
    if score<threshold:
        return "unknown",score
    return tag,score

def chat():
    while True:
        user=input("You: ")
        tag,score=predict_intent_cosine(user)
        print(f"[NLP detected intent: {tag}, confidence: {score:.2f}]")
        if tag in responses:
            print("Bot:",random.choice(responses[tag]))
        else:
            print("Bot: I do not understand.")
        if tag=="farewell":
            break

if __name__=="__main__":
    chat()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python

Accuracy: 0.3333333333333333
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       1.00      0.50      0.67         2
           2       0.11      1.00      0.20         1
           3       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         2
           6       0.00      0.00      0.00         1
           7       1.00      1.00      1.00         2
           8       0.00      0.00      0.00         2

    accuracy                           0.33        12
   macro avg       0.26      0.31      0.23        12
weighted avg       0.34      0.33      0.29        12

[NLP detected intent: greeting, confidence: 1.00]
Bot: Hello! Welcome to CollegeBot.
